# Sample Review — Structured JSON

Side-by-side comparison of structured JSON from both models with inline error detection.

| Colour | Meaning |
|---|---|
| 🔴 Red header | Document title |
| 🔵 Blue | Section title |
| 🟡 Yellow | Question text |
| Dark + green border | Answer text |
| 🟠 Orange + ⚠ | Suspicious label (looks like body text mis-classified) |
| 🔴 Dark red + ✕ | Missing / empty content |

In [1]:
from pathlib import Path
import json, re
from html import escape
from IPython.display import HTML, display

ROOT        = Path("..") if (Path("..") / "data").exists() else Path(".")
LLM_DIR     = ROOT / "data" / "llmlabeled"
MANUAL_DIR  = ROOT / "data" / "manuallabeled"

MODELS = ["llama3.3-70b", "llama3.1-8b"]
MODEL_COLORS = {
    "llama3.3-70b": "#7048e8",
    "llama3.1-8b":  "#1098ad",
}

SAMPLES = sorted(
    [p.stem.replace("_llama3.3-70b_structured", "")
     for p in LLM_DIR.glob("*_llama3.3-70b_structured.json")],
    key=lambda s: int(s.replace("sample", ""))
)
print(f"Found {len(SAMPLES)} samples: {SAMPLES}")

Found 10 samples: ['sample1', 'sample2', 'sample3', 'sample4', 'sample5', 'sample6', 'sample7', 'sample8', 'sample9', 'sample10']


In [ ]:
# ── Error detection ───────────────────────────────────────────────────────────
# Matches list items like (i), (ii), (iii) — body text mis-labeled as section title
_ROMAN_RE = re.compile(r'^\(i{1,3}v?i*\)', re.IGNORECASE)

def _suspicious_section(text: str) -> str | None:
    if not text:
        return "Empty section title"
    if text[0].islower():
        return "Starts with lowercase"
    if _ROMAN_RE.match(text):
        return "List item in section title"
    if text.endswith(".") and not re.search(r'\d\.\s', text):
        return "Ends with period"
    return None

def _suspicious_question(text: str) -> str | None:
    if not text:
        return None
    if text[0].islower():
        return "Starts with lowercase"
    return None

def _struct_counts(data: dict) -> tuple[int, int]:
    t = data["narrative"]["template"]
    sections  = t.get("section", [])
    questions = sum(len(s.get("question", [])) for s in sections)
    return len(sections), questions

def count_errors(data: dict) -> int:
    t = data["narrative"]["template"]
    n = 0
    if not t.get("title"):
        n += 1
    for s in t.get("section", []):
        if _suspicious_section(s.get("title", "").strip()):
            n += 1
        if not s.get("question"):
            n += 1
        for q in s.get("question", []):
            if _suspicious_question(q.get("text", "").strip()):
                n += 1
            if not q.get("answer", {}).get("json", {}).get("answer", ""):
                n += 1
    return n

# ── Shared HTML helpers ───────────────────────────────────────────────────────

def _badge(msg: str, kind: str = "warn") -> str:
    bg   = "#f59f00" if kind == "warn" else "#e03131"
    icon = "⚠" if kind == "warn" else "✕"
    return (
        f'<span style="background:{bg};color:white;font-size:10px;'
        f'padding:2px 6px;border-radius:3px;margin-left:6px;'
        f'font-weight:bold;vertical-align:middle;">{icon} {escape(msg)}</span>'
    )

def _answer_block(answer: str, check_empty: bool = True) -> str:
    if answer:
        return (
            f'<div style="background:#161b22;color:#e6edf3;padding:10px 12px;'
            f'margin-bottom:6px;border-left:5px solid #51cf66;'
            f'white-space:pre-wrap;border-radius:4px;font-size:12px;">'
            f'{escape(answer)}</div>'
        )
    if check_empty:
        return (
            f'<div style="background:#3b0000;color:#ffa8a8;padding:8px 12px;'
            f'margin-bottom:6px;border-left:5px solid #e03131;'
            f'border-radius:4px;font-size:12px;">✕ Empty answer</div>'
        )
    return ""

def _render_sections(template: dict, error_check: bool = False) -> str:
    parts = []
    for section in template.get("section", []):
        sec_title = section.get("title", "").strip()
        sec_warn  = _suspicious_section(sec_title) if error_check else None

        sec_bg = "#e67700" if sec_warn else "#4dabf7"
        parts.append(
            f'<div style="background:{sec_bg};color:white;padding:8px 12px;'
            f'font-weight:bold;border-radius:6px;margin-top:12px;font-size:13px;">'
            f'{escape(sec_title) if sec_title else "(empty)"}'
            f'{_badge(sec_warn) if sec_warn else ""}</div>'
        )

        sec_desc = section.get("description", "").strip()
        if sec_desc:
            parts.append(
                f'<div style="background:#161b22;color:#c9d1d9;padding:10px 12px;'
                f'margin-bottom:8px;border-left:5px solid #4dabf7;'
                f'white-space:pre-wrap;border-radius:4px;font-size:12px;">'
                f'{escape(sec_desc)}</div>'
            )

        questions = section.get("question", [])
        if not questions and error_check:
            parts.append(
                f'<div style="background:#3b0000;color:#ffa8a8;padding:8px 12px;'
                f'border-left:5px solid #e03131;border-radius:4px;'
                f'font-size:12px;margin-top:6px;">✕ No questions in this section</div>'
            )

        for question in questions:
            q_text = question.get("text", "").strip()
            answer = question.get("answer", {}).get("json", {}).get("answer", "")
            q_warn = _suspicious_question(q_text) if error_check else None

            if q_text:
                q_bg = "#e67700" if q_warn else "#ffd43b"
                q_fg = "white"   if q_warn else "black"
                parts.append(
                    f'<div style="background:{q_bg};color:{q_fg};padding:7px 10px;'
                    f'margin-top:8px;border-radius:4px;font-weight:bold;font-size:12px;">'
                    f'{escape(q_text)}{_badge(q_warn) if q_warn else ""}</div>'
                )
            else:
                style = (
                    'background:#332b00;color:#ffe066;' if error_check
                    else 'background:#1c2a1c;color:#8fbc8f;'
                )
                parts.append(
                    f'<div style="{style}padding:5px 10px;margin-top:8px;'
                    f'border-radius:4px;font-size:11px;font-style:italic;">'
                    f'(implicit question — no question text)</div>'
                )

            parts.append(_answer_block(answer, check_empty=error_check))

    return "".join(parts)

# ── Manual column renderer ────────────────────────────────────────────────────

def render_manual(data: dict) -> str:
    template = data["narrative"]["template"]
    parts    = []
    parts.append(
        f'<div style="background:#2f9e44;color:white;padding:7px 12px;'
        f'font-size:12px;font-weight:bold;border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;margin-bottom:10px;">Manual Label (ground truth)</div>'
    )
    title = template.get("title", "").strip()
    parts.append(
        f'<div style="background:#ff6b6b;color:white;padding:10px 12px;'
        f'font-size:15px;font-weight:bold;border-radius:6px;margin-bottom:10px;">'
        f'{escape(title) if title else "(no title)"}</div>'
    )
    parts.append(_render_sections(template, error_check=False))
    return "".join(parts)

# ── Generated column renderer ─────────────────────────────────────────────────

def render_generated(data: dict, model_tag: str, manual_data: dict) -> str:
    template     = data["narrative"]["template"]
    color        = MODEL_COLORS.get(model_tag, "#555")
    n_errors     = count_errors(data)
    m_secs, m_qs = _struct_counts(manual_data)
    g_secs, g_qs = _struct_counts(data)
    parts        = []

    pill = (
        f'<span style="background:#e03131;color:white;font-size:11px;'
        f'padding:2px 8px;border-radius:10px;margin-left:8px;">'
        f'{n_errors} issue{"s" if n_errors != 1 else ""}</span>'
        if n_errors else
        f'<span style="background:#2f9e44;color:white;font-size:11px;'
        f'padding:2px 8px;border-radius:10px;margin-left:8px;">✓ clean</span>'
    )
    parts.append(
        f'<div style="background:{color};color:white;padding:7px 12px;'
        f'font-size:12px;font-weight:bold;border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;margin-bottom:6px;display:flex;align-items:center;">'
        f'{model_tag}{pill}</div>'
    )

    def _diff_pill(label, gen, man):
        ok = gen == man
        bg = "#2f9e44" if ok else "#e03131"
        return (
            f'<span style="background:{bg};color:white;font-size:10px;'
            f'padding:2px 7px;border-radius:4px;margin-right:5px;">'
            f'{label}: {gen} {"✓" if ok else f"≠ {man}"}</span>'
        )

    parts.append(
        f'<div style="padding:5px 0 10px 0;display:flex;flex-wrap:wrap;gap:4px;">'
        f'{_diff_pill("Sections", g_secs, m_secs)}'
        f'{_diff_pill("Questions", g_qs, m_qs)}'
        f'</div>'
    )

    title = template.get("title", "").strip()
    parts.append(
        f'<div style="background:#ff6b6b;color:white;padding:10px 12px;'
        f'font-size:15px;font-weight:bold;border-radius:6px;margin-bottom:10px;">'
        f'{escape(title) if title else "(no title)"}'
        f'{_badge("Missing title", "error") if not title else ""}</div>'
    )

    parts.append(_render_sections(template, error_check=True))
    return "".join(parts)

In [3]:
# Change SAMPLE to a specific name (e.g. "sample1") or keep "all"
SAMPLE = "all"

to_show = SAMPLES if SAMPLE == "all" else [SAMPLE]

for sample in to_show:
    manual_path = MANUAL_DIR / f"{sample}_dmp.json"
    if not manual_path.exists():
        print(f"Skipping {sample} — no manual label found")
        continue

    manual_data = json.loads(manual_path.read_text(encoding="utf-8"))
    manual_col  = render_manual(manual_data)

    gen_cols = []
    total_issues = 0
    for model in MODELS:
        path = LLM_DIR / f"{sample}_{model}_structured.json"
        if path.exists():
            data = json.loads(path.read_text(encoding="utf-8"))
            gen_cols.append(render_generated(data, model, manual_data))
            total_issues += count_errors(data)
        else:
            gen_cols.append(f'<div style="color:#888;padding:20px;">Not found: {path.name}</div>')

    s_color = "#e03131" if total_issues else "#2f9e44"
    s_text  = f"{total_issues} issue{'s' if total_issues != 1 else ''}" if total_issues else "✓ no issues"

    display(HTML(
        f'<details open>'
        f'<summary style="font-size:17px;font-weight:bold;padding:10px;'
        f'cursor:pointer;display:flex;align-items:center;gap:10px;">'
        f'{sample}'
        f'<span style="background:{s_color};color:white;font-size:11px;'
        f'padding:2px 10px;border-radius:10px;font-weight:normal;">{s_text}</span>'
        f'</summary>'
        f'<div style="display:flex;gap:10px;background:#0d1117;padding:16px;'
        f'border-radius:8px;max-height:900px;overflow:auto;">'
        f'<div style="flex:1;min-width:0;">{manual_col}</div>'
        f'<div style="width:1px;background:#30363d;"></div>'
        f'<div style="flex:1;min-width:0;">{gen_cols[0]}</div>'
        f'<div style="width:1px;background:#30363d;"></div>'
        f'<div style="flex:1;min-width:0;">{gen_cols[1]}</div>'
        f'</div></details>'
    ))